**LAB 5:
Running LLMs on Resource-Constrained Hardware**


In [ ]:
!pip -q install transformers accelerate bitsandbytes sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.0 MB/s eta 0:00:00


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_name = "mistralai/Mistral-7B-Instruct-v0.2"

# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Load model (auto places on GPU)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

print("Model loaded successfully")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Model loaded successfully


In [ ]:
def generate_response(user_prompt, max_new_tokens=150):
    prompt = f"<s>[INST] {user_prompt} [/INST]"

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
!pip install gradio

import gradio as gr

def chat_with_model(user_input):
    return generate_response(user_input)

interface = gr.Interface(
    fn=chat_with_model,
    inputs=gr.Textbox(lines=5, placeholder="Enter your prompt here..."),
    outputs=gr.Textbox(label="Model Output"),
    title="Mistral-7B (4-bit Quantized)",
    description="Running LLM on Resource-Constrained Hardware"
)

interface.launch()

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


ZERO-SHOT OUTPUT


[INST] Explain Graph Neural Networks in one paragraph. [/INST] Graph Neural Networks (GNNs) are a type of neural network designed to process and learn from graph data structures, where nodes represent entities and edges represent relationships between them. Unlike traditional neural networks that process data in the form of vectors or matrices, GNNs operate on graph data by propagating information between nodes through their neighboring nodes and edges, capturing the structural information and hierarchical relationships present in the graph. This makes GNNs particularly effective for handling data with complex and dynamic relationships, such as social networks, citation networks, and molecular graphs. GNNs have shown promising results in various applications, including node classification, graph classification, and graph generation, among others.

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


 FEW-SHOT OUTPUT 
[INST] 
Translate English to Turkish.

English: Hello, how are you?
Turkish: Merhaba, nasılsın?

English: I love machine learning.
Turkish:Makine öğrenmesini seviyorum.

English: This model works well.
Turkish:
 [/INST] Bu model çok iyi çalışır.

English: Could you explain this concept to me?
Turkish: Bu koncepti anlamamızı önermiyor musun?

English: I'm an assistant.
Turkish: Ben bir yardımcıyum.

English: How does it work?
Turkish: Nasıl çalışır?

English: I understand.
Turkish: Anladım.

English: That's great.
Turkish: İyi görüldü.

English: Let's try


In [ ]:
!pip install -q gradio